Kawasaki Quantum Summer Camp 2026

# IBM Quantum の本物の量子コンピューターで計算する

Kifumi Numata, IBM Quantum (Aug 02, 2026)

まず必要なライブラリーをインストールします。

In [ ]:
%pip install 'qiskit[visualization]' qiskit-ibm-runtime qiskit-aer

# 1. IBM Quantumで実行するための準備

実量子コンピューターで実験するため次の手順で、API keyとCRNを Colab のシークレットに登録します。

1) https://quantum.cloud.ibm.com/ にサインインし、左側 「API key」の横にある「Create +」をクリックします。
![](https://raw.githubusercontent.com/quantum-tokyo/kawasaki-quantum-camp/refs/heads/main/day2/images/1_iqp.jpg)

2) API keyの名前（例：my API など）を自由に入力し、「Create」をクリックします。
![](https://github.com/quantum-tokyo/kawasaki-quantum-camp/blob/main/day2/images/2_create_api.jpg?raw=true)

3) 「Download」をクリックして「apikey.json」ファイルを保存します。
![](https://github.com/quantum-tokyo/kawasaki-quantum-camp/blob/main/day2/images/3_download_api.jpg?raw=true)

4) 先ほど保存した「apikey.json」ファイルから、apikey をコピーします。
![](https://github.com/quantum-tokyo/kawasaki-quantum-camp/blob/main/day2/images/4_copy_api.jpg?raw=true)

5) Colab の左側の鍵アイコン「シークレット」を開き、「＋新しいシークレットを追加」→ 名前「IBM_QUANTUM_API」を入力 → 値に apikey を貼り付け →「ノートブックからのアクセス」をオンにします。
![](https://github.com/quantum-tokyo/kawasaki-quantum-camp/blob/main/day2/images/5_secrets_api.jpg?raw=true)

6) CRN をコピーします。
![](https://github.com/quantum-tokyo/kawasaki-quantum-camp/blob/main/day2/images/6_crn.jpg?raw=true)

8) Colab の左側の鍵アイコン「シークレット」から「＋新しいシークレットを追加」→ 名前「IBM_QUANTUM_CRN」を入力 → 値に CRN を貼り付け →「ノートブックからのアクセス」をオンにします。
![](https://github.com/quantum-tokyo/kawasaki-quantum-camp/blob/main/day2/images/7_secrets_crn.jpg?raw=true)

続けて以下を実行してください。使うことのできる量子コンピューターのデバイスが個表示されます。

In [ ]:
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

api_key = userdata.get("IBM_QUANTUM_API")
crn     = userdata.get("IBM_QUANTUM_CRN")

service = QiskitRuntimeService(
    channel="ibm_cloud",
    token=api_key,
    instance=crn,
)
service.backends()

In [ ]:
# 以下でデバイスを指定できます。
backend = service.backend('ibm_fez') 

In [ ]:
#一番空いているバックエンドを自動的に選択することもできます
backend = service.least_busy(operational=True)
print("最も空いているバックエンドは: ", backend)

# 2. ベル状態の実験
2量子ビットのもつれ状態を実験します。

In [ ]:
# ２量子ビット回路を作成します。
from qiskit import QuantumCircuit
qc = QuantumCircuit(2,2)    # 2量子ビット, 2古典ビットレジスター

# ゲートを適用します。
qc.h(0)
qc.cx(0,1)    # 制御NOTゲート

# 測定ゲートを追加
qc.measure(0,0)    # 量子ビットq0を測定して、古典レジスターc0に入れます
qc.measure(1,1)    # 量子ビットq1を測定して、古典レジスターc1に入れます

# 回路を描画
qc.draw(output="mpl")

In [ ]:
# 実機のバックエンドでの実行に最適な回路に変換します
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
isa_circuit.draw("mpl", idle_wires=False)

In [ ]:
# Samplerで実行します
from qiskit_ibm_runtime import SamplerV2
sampler = SamplerV2(backend)
job = sampler.run([isa_circuit])

print("job id:", job.job_id()) # 実行に時間がかかるのでjob_idを表示します

In [ ]:
#job = service.job(job.job_id()) 
job = service.job("d1mbhan29o4s73aqp3c0") 
job.status() # ジョブの実行状態を確認します

上記のセルを何回か実行して、'DONE' が表示されたら、実機での実行が終わっているので、以下のセルを実行して結果を確認します。

In [ ]:
### 'DONE'になってから実行します ###
result = job.result()
print(result[0].data.c.get_counts())

In [ ]:
from qiskit.visualization import plot_histogram
plot_histogram(result[0].data.c.get_counts())

# 3. 量子テレポーテーションの実験

In [ ]:
# 3量子ビット回路を用意
from qiskit import QuantumCircuit
qc = QuantumCircuit(3,3)  

# Aliceのもつ未知の量子状態ψをRxで作ります。角度はπ/3にしました。
import numpy as np
qc.rx(np.pi/3,0)
qc.barrier()    #回路を見やすくするために入れます

# 量子もつれを作ります
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# AliceがCNOTとHで自分の量子ビット2つをエンタングルさせ測定します。
qc.cx(0, 1)
qc.h(0)
qc.barrier()
qc.measure(0, 0)
qc.measure(1, 1)

#Aliceが測定結果をBobに送り、Bobが結果に合わせて操作します
with qc.if_test((1, 1)): # 古典レジスター1の値が1だったらXゲートをq2にかける
    qc.x(2)
with qc.if_test((0, 1)): # 古典レジスター0の値が1だったらZゲートをq2にかける
    qc.z(2)

# 未知の量子状態ψの逆ゲートをかけて０が測定できるか確かめます
qc.rx(-np.pi/3, 2)    
qc.measure(2, 2)

qc.draw(output="mpl")

In [ ]:
# シミュレーターで実験
from qiskit_aer import AerSimulator
backend_sim = AerSimulator()

from qiskit_ibm_runtime import SamplerV2
sampler = SamplerV2(backend_sim)
job = sampler.run([qc])
result = job.result()

#  測定された回数を表示
counts = result[0].data.c.get_counts()
print(counts)

# ヒストグラムで測定された確率をプロット
from qiskit.visualization import plot_histogram
plot_histogram( counts )

In [ ]:
# 実機のバックエンドでの実行に最適な回路に変換します
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
isa_circuit.draw("mpl", idle_wires=False)

In [ ]:
# Samplerで実行します
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution_path" : "gen3-experimental"}
job = sampler.run([isa_circuit])

print("job id:", job.job_id()) # job idの確認

In [ ]:
# ジョブの実行状態を確認します
job = service.job('d21k1fbni11c73fdnj9g') # 例です。上に出力された自分のjob_idを入れて実行してください。
job.status()

In [ ]:
### DONEになってから実行します ###
result = job.result()
print(f" > Counts: {result[0].data.c.get_counts()}")

In [ ]:
plot_histogram(result[0].data.c.get_counts())

In [ ]:
# Qiskitバージョンの確認
import qiskit
qiskit.__version__